# 02 · SQL analysis: Paris apartment prices 2021–2025

Each section answers one business question. The SQL lives in `sql/03_analysis.sql` and is run by `src/build_database.py`, which stores each answer as an `agg_*` table and exports it to `data/processed/` for Power BI.
This notebook reads those tables, charts them, and writes one insight per question.

> Run `python src/build_database.py` first. Data: 140k cleaned apartment sales (see the cleaning funnel below). All prices are **medians in € per m²**.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "paris_dvf.duckdb"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH), read_only=True)

def q(sql: str):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql).df()

BLUE = "#3b6ea5"
euro = mtick.FuncFormatter(lambda x, _: f"{x:,.0f} €")
sns.set_theme(style="whitegrid")

## Cleaning funnel (from Phase 3)

In [ ]:
q('SELECT * FROM cleaning_funnel ORDER BY step_order')

## Q1. What is the median price per m² in each arrondissement, each year?
SQL: `MEDIAN`, `QUANTILE_CONT` grouped by arrondissement and year.

In [ ]:
q1 = q("SELECT * FROM agg_median_by_arr_year")
heat = q1.pivot(index="arrondissement", columns="year", values="median_price_m2")

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(heat, annot=True, fmt=",.0f", cmap="Blues", cbar_kws={"label": "Median € per m²"}, ax=ax)
ax.set_title("Median price per m² by arrondissement and year")
ax.set_xlabel("Year"); ax.set_ylabel("Arrondissement")
fig.tight_layout(); fig.savefig(FIG_DIR / "q1_median_by_arr_year.png", dpi=150); plt.show()
heat

**Insight:** _to be written from the real output above._

## Q2. How have prices moved year over year?
SQL: `LAG` over a window partitioned by arrondissement.

In [ ]:
paris = q("SELECT * FROM agg_paris_by_year")
paris

In [ ]:
yoy = q("SELECT * FROM agg_yoy_change WHERE yoy_change_pct IS NOT NULL")
yoy_heat = yoy.pivot(index="arrondissement", columns="year", values="yoy_change_pct")

fig, ax = plt.subplots(figsize=(7, 8))
sns.heatmap(yoy_heat, annot=True, fmt=".1f", cmap="RdBu", center=0, cbar_kws={"label": "YoY change (%)"}, ax=ax)
ax.set_title("Year-over-year change in median price per m² (%)")
ax.set_xlabel("Year"); ax.set_ylabel("Arrondissement")
fig.tight_layout(); fig.savefig(FIG_DIR / "q2_yoy_change.png", dpi=150); plt.show()

**Insight:** _to be written from the real output above._

## Q3. Which arrondissements gained or lost the most over the period?
SQL: `FIRST_VALUE` / `LAST_VALUE` over the full window frame, then `RANK`.

In [ ]:
q3 = q("SELECT * FROM agg_period_change ORDER BY change_pct")
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#b2443a" if v < 0 else BLUE for v in q3["change_pct"]]
ax.barh(q3["arrondissement"].astype(str), q3["change_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f"Change in median price per m², {q3['first_year'].min()} → {q3['last_year'].max()}")
ax.set_xlabel("Change"); ax.set_ylabel("Arrondissement")
fig.tight_layout(); fig.savefig(FIG_DIR / "q3_period_change.png", dpi=150); plt.show()
q3.sort_values("rank_gainer")

**Insight:** _to be written from the real output above._

## Q4. What is the short-term trend? Rolling 3-month median
SQL: `MEDIAN(...) OVER (ORDER BY month RANGE BETWEEN INTERVAL 2 MONTH PRECEDING AND CURRENT ROW)`, computed over all sales in the window.

In [ ]:
q4 = q("SELECT * FROM agg_rolling_3m")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q4["month"], q4["median_price_m2"], color=BLUE, alpha=0.35, label="Monthly median")
ax.plot(q4["month"], q4["rolling_3m_median_m2"], color=BLUE, linewidth=2.5, label="Rolling 3-month median")
ax.yaxis.set_major_formatter(euro)
ax.set_title("Paris median price per m², monthly and rolling 3 months")
ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR / "q4_rolling_3m.png", dpi=150); plt.show()

peak = q4.loc[q4["rolling_3m_median_m2"].idxmax()]
low_after_peak = q4[q4["month"] > peak["month"]].nsmallest(1, "rolling_3m_median_m2").iloc[0]
latest = q4.iloc[-1]
print(f"Peak:   {peak['month']:%Y-%m}  {peak['rolling_3m_median_m2']:,.0f} €/m²")
print(f"Low:    {low_after_peak['month']:%Y-%m}  {low_after_peak['rolling_3m_median_m2']:,.0f} €/m²  ({low_after_peak['rolling_3m_median_m2'] / peak['rolling_3m_median_m2'] - 1:+.1%} vs peak)")
print(f"Latest: {latest['month']:%Y-%m}  {latest['rolling_3m_median_m2']:,.0f} €/m²")

**Insight:** _to be written from the real output above._

## Q5. Is there seasonality in sales volume?
SQL: `DATE_TRUNC('quarter', ...)` and each quarter's share of its year.

In [ ]:
q5 = q("SELECT * FROM agg_sales_by_quarter")
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q5["quarter"].dt.strftime("%Y-Q") + q5["quarter_of_year"].astype(str), q5["sales"], color=BLUE)
ax.set_title("Apartment sales per quarter"); ax.set_ylabel("Sales")
plt.xticks(rotation=60); fig.tight_layout(); fig.savefig(FIG_DIR / "q5_sales_by_quarter.png", dpi=150); plt.show()

q5.groupby("quarter_of_year")["share_of_year_pct"].mean().round(1).to_frame("avg_share_of_year_pct")

**Insight:** _to be written from the real output above._

## Q6. Does price per m² depend on apartment size?
SQL: `CASE WHEN` size buckets, then `PIVOT` with one column per year.

In [ ]:
q6 = q("SELECT * FROM agg_size_bucket_pivot ORDER BY size_bucket")
sizes = q("SELECT size_bucket, SUM(sales) AS sales FROM agg_size_bucket GROUP BY 1 ORDER BY 1")
q6.merge(sizes, on="size_bucket")

In [ ]:
q6_long = q("SELECT * FROM agg_size_bucket")
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=q6_long, x="year", y="median_price_m2", hue="size_bucket", marker="o", palette="Blues_r", ax=ax)
ax.yaxis.set_major_formatter(euro); ax.set_xticks(sorted(q6_long["year"].unique()))
ax.set_title("Median price per m² by apartment size"); ax.set_xlabel("Year"); ax.set_ylabel("")
ax.legend(title="Size", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout(); fig.savefig(FIG_DIR / "q6_size_buckets.png", dpi=150); plt.show()

**Insight:** _to be written from the real output above._

## Q7. Which arrondissements have the widest price spread?
SQL: `STDDEV_SAMP` and IQR (`QUANTILE_CONT(0.75) - QUANTILE_CONT(0.25)`), with IQR as a % of the median so cheap and expensive areas are comparable.

In [ ]:
q7 = q("SELECT * FROM agg_volatility ORDER BY iqr_pct_of_median")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(q7["arrondissement"].astype(str), q7["iqr_pct_of_median"], color=BLUE)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Price spread: IQR as % of median price per m²"); ax.set_xlabel("IQR / median"); ax.set_ylabel("Arrondissement")
fig.tight_layout(); fig.savefig(FIG_DIR / "q7_volatility.png", dpi=150); plt.show()
q7.sort_values("iqr_pct_of_median", ascending=False)

**Insight:** _to be written from the real output above._

## Q8. Which areas are priced below what the model predicts?
_Added after Phase 6 (model)._

## Key findings

_3 to 5 findings with real numbers, written from the outputs above. These go into the README._

In [ ]:
con.close()